In [9]:
import requests
import json

url = 'http://localhost:8083/ctakes-web-rest/service/analyze'
params = {'pipeline': 'Default'}
headers = {'cache-control': 'no-cache'}
data = """
The patient is a 67-year-old male presenting to the emergency department with shortness of breath and a persistent cough for the past 48 hours. He reports a subjective fever and general malaise. The patient began experiencing exertional dyspnea two days ago, which has progressed to dyspnea at rest. He has a productive cough with yellow sputum. He denies chest pain or palpitations. He notes that he has had a runny nose for about a week. The patient's history is significant for hypertension diagnosed in 2010, managed with lisinopril 20mg daily. He also has a history of Type 2 Diabetes Mellitus, diagnosed in 2015, for which he takes metformin 1000mg twice a day. No known drug allergies. The patient's vitals are: BP 145/88, HR 95, Temp 100.2°F, RR 22, SpO2 90% on room air. Lungs show diminished breath sounds at the right lower lobe with crackles on auscultation. Cardiovascular is regular rate and rhythm, no murmurs. There is no peripheral edema. The patient's presentation is concerning for community-acquired pneumonia. We will initiate treatment with azithromycin 500mg IV stat and follow with 250mg PO daily. We will also order a chest X-ray and obtain a sputum culture to confirm the diagnosis and identify the causative organism. The patient will be admitted to the hospital for observation and management of his respiratory distress. We will continue his home medications as directed."""

try:
    response = requests.post(url, params=params, headers=headers, data=data)
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    print("Status Code:", response.status_code)
    print("Response Body:")
    print(response.text)

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

Status Code: 200
Response Body:
{"AnatomicalSiteMention":[{"begin":357,"end":362,"text":"chest","polarity":1,"conceptAttributes":[{"code":"51185008","cui":"C0817096","codingScheme":"SNOMEDCT_US","tui":"T029"},{"code":"261179002","cui":"C0817096","codingScheme":"SNOMEDCT_US","tui":"T029"}]},{"begin":1145,"end":1150,"text":"chest","polarity":1,"conceptAttributes":[{"code":"51185008","cui":"C0817096","codingScheme":"SNOMEDCT_US","tui":"T029"},{"code":"261179002","cui":"C0817096","codingScheme":"SNOMEDCT_US","tui":"T029"}]}],"MedicationMention":[{"begin":527,"end":537,"text":"lisinopril","polarity":1,"conceptAttributes":[{"code":"29046","cui":"C0065374","codingScheme":"RXNORM","tui":"T121"},{"code":"108575001","cui":"C0065374","codingScheme":"SNOMEDCT_US","tui":"T121"},{"code":"386873009","cui":"C0065374","codingScheme":"SNOMEDCT_US","tui":"T121"}]},{"begin":639,"end":648,"text":"metformin","polarity":1,"conceptAttributes":[{"code":"6809","cui":"C0025598","codingScheme":"RXNORM","tui":"T12

In [10]:
def pretty_print_ctakes_output(json_output):
    """
    Parses and prints the cTAKES JSON output in a human-readable format.

    Args:
        json_output (dict): The dictionary containing the cTAKES JSON data.
    """
    # Define a user-friendly mapping for the keys
    category_map = {
        "AnatomicalSiteMention": "Anatomical Sites 📍",
        "MedicationMention": "Medications 💊",
        "DiseaseDisorderMention": "Diseases & Disorders 🤒",
        "SignSymptomMention": "Signs & Symptoms 🤕",
        "ProcedureMention": "Procedures 🩺",
        "MeasurementAnnotation": "Measurements 📏",
        # You can add more categories here as needed
    }

    # Iterate through the main categories of the JSON output
    for key, human_name in category_map.items():
        if key in json_output and json_output[key]:
            print(f"--- {human_name} ---")
            
            # Iterate through each mention in the category
            for mention in json_output[key]:
                text = mention.get("text", "N/A")
                polarity = "Positive" if mention.get("polarity", 1) == 1 else "Negative"
                concept_attributes = mention.get("conceptAttributes", [])
                
                codes_and_schemes = []
                for attr in concept_attributes:
                    code = attr.get("code")
                    scheme = attr.get("codingScheme")
                    cui = attr.get("cui")
                    codes_and_schemes.append(f"({scheme}: {code}, CUI: {cui})")
                
                print(f"  - **{text}**")
                print(f"    - Polarity: {polarity}")
                if codes_and_schemes:
                    print(f"    - Codes: {', '.join(codes_and_schemes)}")
            print("\n")

In [4]:
pretty_print_ctakes_output(json.loads(response.text))

--- Anatomical Sites 📍 ---
  - **chest**
    - Polarity: Positive
    - Codes: (SNOMEDCT_US: 51185008, CUI: C0817096), (SNOMEDCT_US: 261179002, CUI: C0817096)
  - **chest**
    - Polarity: Positive
    - Codes: (SNOMEDCT_US: 51185008, CUI: C0817096), (SNOMEDCT_US: 261179002, CUI: C0817096)


--- Medications 💊 ---
  - **lisinopril**
    - Polarity: Positive
    - Codes: (RXNORM: 29046, CUI: C0065374), (SNOMEDCT_US: 108575001, CUI: C0065374), (SNOMEDCT_US: 386873009, CUI: C0065374)
  - **metformin**
    - Polarity: Positive
    - Codes: (RXNORM: 6809, CUI: C0025598), (SNOMEDCT_US: 109081006, CUI: C0025598), (SNOMEDCT_US: 372567009, CUI: C0025598)
  - **drug**
    - Polarity: Negative
    - Codes: (SNOMEDCT_US: 763158003, CUI: C0013227), (SNOMEDCT_US: 373873005, CUI: C0013227), (SNOMEDCT_US: 410942007, CUI: C0013227)
  - **azithromycin**
    - Polarity: Positive
    - Codes: (RXNORM: 18631, CUI: C0052796), (SNOMEDCT_US: 387531004, CUI: C0052796), (SNOMEDCT_US: 96034006, CUI: C0052796)


-

In [11]:
import requests
import json
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../api/.env")

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:8082"
BEARER_TOKEN = os.getenv("API_BEARER_TOKEN", "")

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {BEARER_TOKEN}"
}

# ── Test clinical note ─────────────────────────────────────────────────────────
CLINICAL_TEXT = """Swollen LL, limited mobility, pain and redness over R LL. Possible PE post THR and TKR. 
IV Streptokinase stat
IV NS 500ml run fast
SC Clean 200U stat
Refer to IR for possible embolectomy"""

print("Configuration loaded.")
print(f"  Base URL : {BASE_URL}")
print(f"  Token    : {BEARER_TOKEN[:10]}***" if BEARER_TOKEN else "  Token    : NOT SET")

Configuration loaded.
  Base URL : http://localhost:8082
  Token    : c73e3c54-b***


In [12]:
# ── Block 2: Health Check ─────────────────────────────────────────────────────
print("=" * 60)
print("Health Check (GET / and GET /health)")
print("=" * 60)

for path in ["/", "/health"]:
    r = requests.get(f"{BASE_URL}{path}", headers=HEADERS, timeout=10)
    status = "✓" if r.status_code == 200 else "✗"
    print(f"{status} {path}  →  {r.status_code}  {r.json()}")

Health Check (GET / and GET /health)
✓ /  →  200  {'message': 'CTakes REST Service API'}
✓ /health  →  200  {'status': 'healthy'}


In [13]:
# ── Block 3: Generate Note (Step 1 — LLM Summary) ────────────────────────────
print("=" * 60)
print("Generate Note  (POST /generate/note)")
print("=" * 60)
print("Input:\n", CLINICAL_TEXT, "\n")

r = requests.post(
    f"{BASE_URL}/generate/note",
    headers=HEADERS,
    json={"text": CLINICAL_TEXT},
    timeout=240
)

if r.status_code == 200:
    data = r.json()
    REFINED_TEXT = data["text"]
    print("✓ Refined Summary:\n")
    print(REFINED_TEXT)
    print("\nToken Usage:", data.get("tokens_used", {}))
else:
    print(f"✗ Failed ({r.status_code}):", r.text)
    REFINED_TEXT = CLINICAL_TEXT  # fallback
    print("Using original text as fallback.")

Generate Note  (POST /generate/note)
Input:
 Swollen LL, limited mobility, pain and redness over R LL. Possible PE post THR and TKR. 
IV Streptokinase stat
IV NS 500ml run fast
SC Clean 200U stat
Refer to IR for possible embolectomy 



KeyboardInterrupt: 

In [16]:
# ── Block 4: Generate Terms (Full Pipeline) ───────────────────────────────────
# Uses REFINED_TEXT from Block 3 (or falls back to CLINICAL_TEXT)
print("=" * 60)
print("Generate Terms  (POST /generate/terms)")
print("=" * 60)
print("Input text (first 200 chars):", REFINED_TEXT[:200], "...\n")

r = requests.post(
    f"{BASE_URL}/generate/terms",
    headers=HEADERS,
    json={"text": REFINED_TEXT},
    timeout=300
)

if r.status_code == 200:
    data = r.json()
    terms = data.get("terms", {})

    print("✓ Extracted SNOMED-CT Terms:\n")
    for category in ["anatomical_sites", "procedures", "symptoms", "medications"]:
        items = terms.get(category, [])
        if items:
            print(f"  [{category.upper()}]")
            for item in items:
                print(f"    • {item['term']}  |  {item['code']}")

    diagnosis = terms.get("diagnosis", {})
    for sub in ["communicable_disease", "non_communicable_disease"]:
        items = diagnosis.get(sub, [])
        if items:
            print(f"  [DIAGNOSIS — {sub.replace('_', ' ').upper()}]")
            for item in items:
                print(f"    • {item['term']}  |  {item['code']}")

    print("\nToken Usage:")
    for step, usage in data.get("tokens_used", {}).items():
        print(f"  {step}: in={usage['input_token']}  out={usage['output_token']}")
else:
    print(f"✗ Failed ({r.status_code}):", r.text)

Generate Terms  (POST /generate/terms)
Input text (first 200 chars): The patient presents with a swollen left lower limb, limited mobility, and pain and redness over the right lower limb, raising clinical suspicion for a possible pulmonary embolism. The patient’s medic ...

✓ Extracted SNOMED-CT Terms:

  [ANATOMICAL_SITES]
    • Structure of joint of left lower limb  |  725435005
    • Structure of joint of right lower limb  |  725436006
    • Hip joint structure  |  24136001
    • Knee joint structure  |  49076000
  [PROCEDURES]
    • Total replacement of hip  |  52734007
    • Total knee replacement  |  609588000
    • Removal of embolus  |  71815002
    • Intravenous infusion  |  14152002
    • Subcutaneous injection of heparin  |  312613008
    • Urgent referral  |  134403003
  [SYMPTOMS]
    • Swelling  |  65124004
    • Pain  |  22253000
    • Impaired mobility  |  82971005
  [MEDICATIONS]
    • Streptokinase  |  395889004
    • Sodium chloride  |  387390002
    • Enoxaparin  | 

In [17]:
# ── Block 5: cTAKES Health Check ─────────────────────────────────────────────
print("=" * 60)
print("cTAKES Health Check  (GET /generate/ctakes/health)")
print("=" * 60)

r = requests.get(
    f"{BASE_URL}/generate/ctakes/health",
    headers=HEADERS,
    timeout=120
)

if r.status_code == 200:
    data = r.json()
    alive   = data.get("alive", False)
    status  = data.get("status", "unknown")
    total   = data.get("total_terms", 0)
    error   = data.get("error", None)

    icon = "✓" if alive else "✗"
    print(f"{icon} Status : {status}")
    print(f"  Alive  : {alive}")
    print(f"  Terms  : {total}")
    if error:
        print(f"  Error  : {error}")
else:
    print(f"✗ Failed ({r.status_code}):", r.text)

cTAKES Health Check  (GET /generate/ctakes/health)
✓ Status : alive
  Alive  : True
  Terms  : 42


In [21]:
import requests

def get_mapping(concept_id, mapping_type="icd10"):
    # Map types to RefSet IDs
    refsets = {
        "icd10": "447562003", # Complex map
        "icd9":  "447563008", # Complex map
        "icd11": "1215115004" # ICD-11 MMS Map
    }
    
    target_refset = refsets.get(mapping_type)
    url = f"http://localhost:8080/browser/MAIN/members"
    
    params = {
        'referenceSet': target_refset,
        'referencedComponentId': concept_id,
        'active': 'true'
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        items = response.json().get('items', [])
        
        return [i['additionalFields'].get('mapTarget') for i in items if 'mapTarget' in i['additionalFields']]
    except Exception as e:
        return f"Error: {e}"

# Example usage for ICD-9
snomed_id = "59282003" # Pulmonary embolism
print(f"ICD-9: {get_mapping(snomed_id, 'icd9')}")
print(f"ICD-10: {get_mapping(snomed_id, 'icd10')}")
print(f"ICD-11: {get_mapping(snomed_id, 'icd11')}")

ICD-9: []
ICD-10: ['I26.9']
ICD-11: []


In [22]:
import requests

def list_loaded_refsets():
    url = "http://localhost:8080/fhir/ValueSet/$expand"
    params = {'url': 'http://snomed.info/sct?fhir_vs=refset'}
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        expansion = response.json().get('expansion', {}).get('contains', [])
        
        print(f"{'ID':<18} | {'Display Name'}")
        print("-" * 50)
        for item in expansion:
            # We filter for 'map' to find relevant ones
            if 'map' in item['display'].lower():
                print(f"{item['code']:<18} | {item['display']}")
                
    except Exception as e:
        print(f"Error connecting to Snowstorm: {e}")

list_loaded_refsets()

ID                 | Display Name
--------------------------------------------------
900000000000497000 | CTV3 to SNOMED CT simple map
447562003          | SNOMED CT to ICD-10 extended map
446608001          | SNOMED CT to ICD-O simple map
